In [0]:
from pathlib import Path
import pandas as pd
from pyspark.sql.functions import current_timestamp,regexp_replace, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

In [0]:
## normalizando la maestra de clientes
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
df = spark.table("workspace.silver.superstore").toPandas()


spark.sql("DROP TABLE IF EXISTS workspace.gold.mst_customers")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.mst_customers (
  customer_id   STRING,
  customer_name STRING,
  segment       STRING
)
""")

mst_customers = (
    df.reindex(columns=["customer_id", "customer_name", "segment"])
      .dropna(subset=["customer_id"])
      .drop_duplicates(subset=["customer_id"])
      .sort_values("customer_name", ignore_index=True)
)

spark.createDataFrame(mst_customers).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.mst_customers")



In [0]:

## normalizando la maestra de productos
spark.sql("DROP TABLE IF EXISTS workspace.gold.mst_products")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.mst_products (
  product_id   STRING,
  product_name STRING,
  category     STRING,
  sub_category STRING
)
""")

mst_products = (
    df.reindex(columns=["product_id", "product_name", "category", "sub_category"])
      .dropna(subset=["product_id"])
      .drop_duplicates(subset=["product_id"])
      .sort_values("product_name", ignore_index=True)
)

spark.createDataFrame(mst_products).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.mst_products")



In [0]:

## normalizando la maestra de localizacion

spark.sql("DROP TABLE IF EXISTS workspace.gold.mst_location")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.mst_location (
  postal_code INT,
  city        STRING,
  state       STRING,
  region      STRING,
  country     STRING
)
""")

mst_location = (
    df.reindex(columns=["postal_code", "city", "state", "region", "country"])
      .dropna(subset=["postal_code"])
      .drop_duplicates(subset=["postal_code"])
      .sort_values("state", ignore_index=True)
)

spark.createDataFrame(mst_location).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.mst_location")



In [0]:

## normalizando la maestra calendario

spark.sql("DROP TABLE IF EXISTS workspace.gold.mst_time")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.mst_time (
  date_key     BIGINT,
  date         TIMESTAMP,
  year         INT,
  quarter      INT,
  month        INT,
  month_name   STRING,
  day          INT,
  day_name     STRING,
  week_of_year BIGINT,
  is_weekend   BOOLEAN
)
""")

date_series = pd.to_datetime(df["order_date"], errors="coerce") \
                .dropna().drop_duplicates().sort_values()

mst_time = pd.DataFrame({"date": date_series})
mst_time["date_key"]     = mst_time["date"].dt.strftime("%Y%m%d").astype(int)
mst_time["year"]         = mst_time["date"].dt.year
mst_time["quarter"]      = mst_time["date"].dt.quarter
mst_time["month"]        = mst_time["date"].dt.month
mst_time["month_name"]   = mst_time["date"].dt.month_name()
mst_time["day"]          = mst_time["date"].dt.day
mst_time["day_name"]     = mst_time["date"].dt.day_name()
mst_time["week_of_year"] = mst_time["date"].dt.isocalendar().week.astype(int)
mst_time["is_weekend"]   = mst_time["day_name"].isin(["Saturday", "Sunday"])

mst_time = mst_time[["date_key", "date", "year", "quarter", "month",
                      "month_name", "day", "day_name", "week_of_year", "is_weekend"]]

spark.createDataFrame(mst_time).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.mst_time")




In [0]:
## normalizando la tabla de hechos

spark.sql("DROP TABLE IF EXISTS workspace.gold.fact_orders")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_orders (
  order_id             STRING,
  customer_id          STRING,
  product_id           STRING,
  postal_code          INT,
  date_key             BIGINT,
  ship_mode            STRING,
  retail_sales_people  STRING,
  returned             STRING,
  sales                DOUBLE,
  quantity             INT,
  discount             DOUBLE,
  profit               DOUBLE
)
""")

dates = pd.to_datetime(df["order_date"], errors="coerce")

fact_orders = (
    df.reindex(columns=[
        "order_id", "customer_id", "product_id", "postal_code",
        "ship_mode", "retail_sales_people", "returned",
        "sales", "quantity", "discount", "profit"
    ])
    .assign(date_key=dates.dt.strftime("%Y%m%d").astype("Int64"))
    .reindex(columns=[
        "order_id", "customer_id", "product_id", "postal_code", "date_key",
        "ship_mode", "retail_sales_people", "returned",
        "sales", "quantity", "discount", "profit"
    ])
    .dropna(subset=["order_id", "product_id"])
)

# Convert numeric columns to proper types
fact_orders["sales"] = fact_orders["sales"].astype(float)
fact_orders["discount"] = fact_orders["discount"].astype(float)
fact_orders["profit"] = fact_orders["profit"].astype(float)

spark.createDataFrame(fact_orders).write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_orders")

In [0]:
tablas=["mst_customers", "mst_products", "mst_location", "mst_time", "fact_orders"]

In [0]:
schema = StructType([
    StructField("nombre_capa", StringType(), True),
    StructField("nombre_tabla", StringType(), True),
    StructField("cnt_registros", IntegerType(), True)
])

rows = []
for tabla in tablas:
    cnt = spark.table(f"workspace.gold.{tabla}").count()
    rows.append(("gold", tabla, cnt))

df_control = spark.createDataFrame(rows, schema) \
    .withColumn("fecha_actualizacion", current_timestamp())

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ctl")

# Creamos la tabla de control si no existe
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.ctl.control_procesos (
    nombre_capa STRING,
    nombre_tabla STRING,
    cnt_registros INT,
    fecha_actualizacion TIMESTAMP
)
USING DELTA
""")

# Insertamos el registro en la tabla de control
df_control.write.format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('workspace.ctl.control_procesos')

display(df_control)

In [0]:
for table in ["mst_customers", "mst_products", "mst_location", "mst_time", "fact_orders"]:
    count = spark.table(f"workspace.gold.{table}").count()
    print(f"workspace.gold.{table}: {count} filas")